# grid_py Tutorial 2: WebRenderer Interactive Output

The **WebRenderer** produces interactive HTML with SVG + Canvas rendering and D3.js interactions.
The API is **identical** to CairoRenderer — just swap the renderer.

All layout uses the **Unit system** (`lines`, `cm`, `npc`) to adapt to different devices.

In [ ]:
import numpy as np
from IPython.display import display

from grid_py import (
    WebRenderer, Gpar, Unit, Viewport, GridLayout,
    get_state, grid_draw,
    push_viewport, pop_viewport,
    rect_grob, circle_grob, text_grob, lines_grob,
    points_grob, polygon_grob, segments_grob, roundrect_grob,
)

## 1. Scatter Plot with Tooltip Data

Hover over points to see species and measurements. Scroll to zoom, drag to pan.

In [ ]:
np.random.seed(42)

# Simulated iris-like data
n_per = 25
species = ['setosa'] * n_per + ['versicolor'] * n_per + ['virginica'] * n_per
petal_length = np.concatenate([
    np.random.normal(1.5, 0.3, n_per),
    np.random.normal(4.3, 0.5, n_per),
    np.random.normal(5.5, 0.5, n_per),
])
petal_width = np.concatenate([
    np.random.normal(0.3, 0.1, n_per),
    np.random.normal(1.3, 0.2, n_per),
    np.random.normal(2.0, 0.3, n_per),
])
color_map = {'setosa': '#e63946', 'versicolor': '#457b9d', 'virginica': '#2a9d8f'}
cols = [color_map[s] for s in species]

r = WebRenderer(width=7, height=5, dpi=100)
state = get_state()
state.init_device(r)

# Background
grid_draw(rect_grob(x=0.5, y=0.5, width=1.0, height=1.0,
                     gp=Gpar(fill='white', col='#ccc')))

# Title: 2 lines at top
title_vp = Viewport(name='title',
    x=Unit(0.5, 'npc'), y=Unit(1, 'npc') - Unit(1, 'lines'),
    width=Unit(1, 'npc'), height=Unit(2, 'lines'))
push_viewport(title_vp, recording=False)
grid_draw(text_grob(label='Iris Scatter (hover for tooltip)', x=0.5, y=0.5,
                     gp=Gpar(fontsize=14, fontface='bold', col='#264653')))
pop_viewport(1, recording=False)

# Plot area with unit-based margins
plot_vp = Viewport(name='plot',
    x=Unit(0.5, 'npc') + Unit(1, 'lines'),
    y=Unit(0.5, 'npc') - Unit(0.5, 'lines'),
    width=Unit(1, 'npc') - Unit(4, 'lines'),
    height=Unit(1, 'npc') - Unit(5, 'lines'))
push_viewport(plot_vp, recording=False)

# Normalize data to NPC within the plot area
x_npc = 0.02 + 0.96 * (petal_length - petal_length.min()) / (petal_length.max() - petal_length.min())
y_npc = 0.02 + 0.96 * (petal_width - petal_width.min()) / (petal_width.max() - petal_width.min())

# Points with metadata for tooltips
grob = points_grob(x=x_npc.tolist(), y=y_npc.tolist(), pch=19,
                    gp=Gpar(col=cols, fill=cols, fontsize=6, alpha=0.8))
grob.metadata = {
    'species': species,
    'petal_length': [f'{v:.1f}' for v in petal_length],
    'petal_width': [f'{v:.1f}' for v in petal_width],
}
grid_draw(grob)

pop_viewport(1, recording=False)

# Axis labels: positioned using lines units
grid_draw(text_grob(label='Petal Length', x=Unit(0.5, 'npc'),
                     y=Unit(1.5, 'lines'),
                     gp=Gpar(fontsize=11, col='#333')))
grid_draw(text_grob(label='Petal Width',
                     x=Unit(1.5, 'lines'), y=Unit(0.5, 'npc'), rot=90,
                     gp=Gpar(fontsize=11, col='#333')))

# Legend: positioned from right edge using cm
for i, (sp, color) in enumerate(color_map.items()):
    ypos = Unit(1, 'npc') - Unit(3 + i * 1.5, 'lines')
    grid_draw(circle_grob(x=Unit(1, 'npc') - Unit(2.5, 'cm'), y=ypos, r=Unit(0.1, 'cm'),
                           gp=Gpar(fill=color, col=color)))
    grid_draw(text_grob(label=sp,
                         x=Unit(1, 'npc') - Unit(2, 'cm'), y=ypos, hjust=0,
                         gp=Gpar(fontsize=9, col='#333')))

display(r)

## 2. Multi-Panel Layout

In [ ]:
r = WebRenderer(width=7, height=5, dpi=100, theme='light')
state.init_device(r)

# Title
title_vp = Viewport(name='title',
    x=Unit(0.5, 'npc'), y=Unit(1, 'npc') - Unit(1, 'lines'),
    width=Unit(1, 'npc'), height=Unit(2, 'lines'))
push_viewport(title_vp, recording=False)
grid_draw(text_grob(label='Multi-Panel Layout (hover for details)', x=0.5, y=0.5,
                     gp=Gpar(fontsize=15, fontface='bold', col='#2c3e50')))
pop_viewport(1, recording=False)

# 1x3 layout with unit margins
layout = GridLayout(nrow=1, ncol=3,
                     widths=Unit([1, 1, 1], 'null'),
                     heights=Unit([1], 'null'))

main_vp = Viewport(name='main',
    x=Unit(0.5, 'npc'),
    y=Unit(0.5, 'npc') - Unit(1, 'lines'),
    width=Unit(1, 'npc') - Unit(1, 'cm'),
    height=Unit(1, 'npc') - Unit(4, 'lines'),
    layout=layout)
push_viewport(main_vp, recording=False)

panel_titles = ['Panel A', 'Panel B', 'Panel C']
panel_colors = ['#e63946', '#457b9d', '#2a9d8f']
panel_groups = ['alpha', 'beta', 'gamma']

for col in [1, 2, 3]:
    vp = Viewport(name=f'panel_{col}', layout_pos_row=1, layout_pos_col=col)
    push_viewport(vp, recording=False)

    # Panel background with unit-based inset
    grid_draw(rect_grob(x=0.5, y=0.5,
                         width=Unit(1, 'npc') - Unit(0.3, 'cm'),
                         height=Unit(1, 'npc') - Unit(0.3, 'cm'),
                         gp=Gpar(fill='#f8f9fa', col=panel_colors[col-1], lwd=2)))

    # Random scatter with tooltip metadata
    np.random.seed(col)
    px = np.random.rand(20) * 0.7 + 0.15
    py = np.random.rand(20) * 0.6 + 0.15
    values = np.random.normal(100, 15, 20)

    grob = points_grob(x=px.tolist(), y=py.tolist(), pch=19,
                        gp=Gpar(col=panel_colors[col-1],
                                fill=panel_colors[col-1], fontsize=5))
    grob.metadata = {
        'panel': [panel_titles[col-1]] * 20,
        'group': [panel_groups[col-1]] * 20,
        'x': [f'{v:.2f}' for v in px],
        'y': [f'{v:.2f}' for v in py],
        'value': [f'{v:.1f}' for v in values],
    }
    grid_draw(grob)

    # Panel title: 1 line from top
    grid_draw(text_grob(label=panel_titles[col-1], x=0.5,
                         y=Unit(1, 'npc') - Unit(1, 'lines'),
                         gp=Gpar(fontsize=11, fontface='bold',
                                 col=panel_colors[col-1])))

    pop_viewport(1, recording=False)

pop_viewport(1, recording=False)

display(r)

## 3. Inspect the Scene Graph

In [ ]:
import json

sg = json.loads(r.to_scene_json())

def print_tree(node, indent=0):
    t = node.get('type', '?')
    name = node.get('name', '')
    hint = node.get('render_hint', '')
    extra = f' name="{name}"' if name else ''
    extra += f' hint={hint}' if hint else ''
    print(' ' * indent + f'{t}{extra}')
    for child in node.get('children', []):
        print_tree(child, indent + 2)

print(f'Scene graph: {sg["width"]}x{sg["height"]}px, {sg["dpi"]} dpi')
print(f'Defs: {len(sg["defs"]["gradients"])} gradients, '
      f'{len(sg["defs"]["clip_paths"])} clips, '
      f'{len(sg["defs"]["masks"])} masks')
print()
print_tree(sg['root'])

## 4. Save as Standalone HTML

In [ ]:
r.save('interactive_plot.html')
print('Saved interactive_plot.html')
print(f'File size: {len(r.to_html()) / 1024:.1f} KB')
print('Open in browser for full interactivity (zoom, pan, tooltips).')